# Conjunto de datos completo sin clusterización

In [1]:
#Importaciones
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler

#Lectura de datos
datos = pd.read_excel('03_Clusterizacion.xlsx')
datos.head(24)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM
0,2022-09-01 00:00:00,0.000000,19,7,77,0,4,15,0,Noche,Noche
1,2022-09-01 01:00:00,0.000000,19,7,82,0,4,16,1,Noche,Noche
2,2022-09-01 02:00:00,0.000000,18,9,85,0,3,16,2,Noche,Noche
3,2022-09-01 03:00:00,0.000000,18,11,87,0,3,16,3,Noche,Noche
4,2022-09-01 04:00:00,0.000000,18,11,88,0,3,16,4,Noche,Noche
5,2022-09-01 05:00:00,0.000000,17,15,86,0,3,14,5,Noche,Noche
6,2022-09-01 06:00:00,0.000000,18,47,89,0,3,16,6,Nublado,Lluvioso
7,2022-09-01 07:00:00,6.584959,18,51,95,0,4,17,7,Nublado,Lluvioso
8,2022-09-01 08:00:00,560.422022,18,47,100,0,3,18,8,Nublado,Lluvioso
9,2022-09-01 09:00:00,7720.582326,18,5,100,1,4,18,9,Nublado,Lluvioso


In [2]:
datos["Generacion_prev_hour"] = datos["Generación"].shift(1)
datos["Generacion_prev_day"] = datos["Generación"].shift(24)
datos = datos.dropna(how="any", axis= 0)

Definimos X y y

In [3]:
datos_dia = datos[datos["Cluster GMM"] == "Lluvioso"].copy()
datos_dia.head(10)

,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day
30,2022-09-02 06:00:00,0.000000,17,7,91,0,4,15,6,Nublado,Lluvioso,0.000000,0.000000
31,2022-09-02 07:00:00,0.000000,17,7,94,0,3,16,7,Nublado,Lluvioso,0.000000,6.584959
32,2022-09-02 08:00:00,438.814997,16,5,97,0,3,15,8,Nublado,Lluvioso,0.000000,560.422022
33,2022-09-02 09:00:00,5908.000884,17,0,93,1,2,16,9,Nublado,Lluvioso,438.814997,7720.582326
34,2022-09-02 10:00:00,5030.740421,18,0,85,2,2,15,10,Nublado,Lluvioso,5908.000884,9433.109309
38,2022-09-02 14:00:00,28500.000000,24,3,37,6,2,9,14,Soleado,Lluvioso,20596.278869,29057.585772
39,2022-09-02 15:00:00,24647.568577,26,7,33,5,4,8,15,Nublado,Lluvioso,28500.000000,30000.000000
40,2022-09-02 16:00:00,25500.000000,27,7,34,4,4,9,16,Nublado,Lluvioso,24647.568577,28062.328964
41,2022-09-02 17:00:00,24281.956494,28,7,36,2,4,12,17,Nublado,Lluvioso,25500.000000,28786.629243
42,2022-09-02 18:00:00,22733.515002,26,7,39,1,4,12,18,Nublado,Lluvioso,24281.956494,29900.303971


In [4]:
columns = datos_dia.drop(columns=["Fecha", "Generación", "Cluster KMeans", "Cluster GMM"]).columns

In [5]:
X = datos_dia[columns]
X

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,17,7,91,0,4,15,6,0.000000,0.000000
31,17,7,94,0,3,16,7,0.000000,6.584959
32,16,5,97,0,3,15,8,0.000000,560.422022
33,17,0,93,1,2,16,9,438.814997,7720.582326
34,18,0,85,2,2,15,10,5908.000884,9433.109309
...,...,...,...,...,...,...,...,...,...
18273,14,0,87,1,4,11,8,67.000000,7302.000000
18274,15,0,83,2,4,12,9,7356.000000,18014.000000
18275,17,0,71,4,3,12,10,17638.000000,23010.000000
18276,19,0,60,5,3,11,11,23339.000000,26156.000000


In [6]:
y = datos_dia[['Generación']]
y

,Generación
30,0.000000
31,0.000000
32,438.814997
33,5908.000884
34,5030.740421
...,...
18273,7356.000000
18274,17638.000000
18275,23339.000000
18276,26323.000000


Dividimos entrenamiento, validación y prueba

In [7]:
train_size = int(0.7 * len(X))
val_size = int(0.85 * len(X))

In [8]:
# Entrenamiento, validación y prueba, 75, 15 y 15
X_train, y_train =  X.iloc[:train_size, :], y.iloc[:train_size, :]
X_val, y_val = X.iloc[train_size:val_size, :], y.iloc[train_size:val_size, :]
X_test, y_test = X.iloc[val_size:, :],  y.iloc[val_size:,:]

print(f'X_train: {len(X_train)}, y_train: {len(y_train)}')
print(f'X_val: {len(X_val)}, y_val: {len(y_val)}')
print(f'X_test: {len(X_test)}, y_test: {len(y_test)}')

X_train: 4263, y_train: 4263
X_val: 913, y_val: 913
X_test: 914, y_test: 914


## Escalar con MinMaxScaler

In [9]:
from sklearn.preprocessing import MinMaxScaler

In [10]:
x_scaler = MinMaxScaler().fit(X_train)
x_scaler

MinMaxScaler()

In [11]:
X_train_scaled = x_scaler.transform(X_train)
print(X_train_scaled)
print(X_train_scaled.shape)

[[4.70588235e-01 7.77777778e-02 9.04255319e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.70588235e-01 7.77777778e-02 9.36170213e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.41176471e-01 5.55555556e-02 9.68085106e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [7.35294118e-01 0.00000000e+00 2.65957447e-01 ... 7.33333333e-01
  8.68733333e-01 6.41966667e-01]
 [6.47058824e-01 0.00000000e+00 3.93617021e-01 ... 8.00000000e-01
  8.07700000e-01 4.83833333e-01]
 [5.88235294e-01 2.22222222e-02 5.21276596e-01 ... 8.66666667e-01
  5.96633333e-01 9.58000000e-02]]
(4263, 9)


In [12]:
X_train_scaled_df = pd.DataFrame(X_train_scaled, index=X_train.index, columns=X_train.columns)
X_train_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.470588,0.077778,0.904255,0.000000,0.75,0.789474,0.000000,0.000000,0.000000
31,0.470588,0.077778,0.936170,0.000000,0.50,0.842105,0.066667,0.000000,0.000219
32,0.441176,0.055556,0.968085,0.000000,0.50,0.789474,0.133333,0.000000,0.018681
33,0.470588,0.000000,0.925532,0.071429,0.25,0.842105,0.200000,0.014627,0.257353
34,0.500000,0.000000,0.840426,0.142857,0.25,0.789474,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
13522,0.382353,0.000000,0.776596,0.142857,0.50,0.578947,0.200000,0.415467,0.612933
13529,0.794118,0.000000,0.202128,0.142857,1.00,0.315789,0.666667,0.862067,0.847600
13530,0.735294,0.000000,0.265957,0.071429,1.00,0.368421,0.733333,0.868733,0.641967
13531,0.647059,0.000000,0.393617,0.071429,1.00,0.526316,0.800000,0.807700,0.483833


In [13]:
X_val_scaled = x_scaler.transform(X_val)
print(X_val_scaled)
print(X_val_scaled.shape)

[[0.55882353 0.48888889 0.61702128 ... 0.93333333 0.11096667 0.        ]
 [0.5        0.56666667 0.72340426 ... 1.         0.         0.        ]
 [0.38235294 0.22222222 0.82978723 ... 0.         0.         0.        ]
 ...
 [0.58823529 0.56666667 0.65957447 ... 0.33333333 0.89683333 0.92476667]
 [0.64705882 0.52222222 0.57446809 ... 0.4        0.92476667 0.93596667]
 [0.67647059 0.44444444 0.5106383  ... 0.46666667 0.9375     0.9438    ]]
(913, 9)


In [14]:
X_val_scaled_df = pd.DataFrame(X_val_scaled, index=X_val.index, columns=X_val.columns)
X_val_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
13533,0.558824,0.488889,0.617021,0.000000,1.0,0.684211,0.933333,0.110967,0.000000
13534,0.500000,0.566667,0.723404,0.000000,1.0,0.736842,1.000000,0.000000,0.000000
13543,0.382353,0.222222,0.829787,0.000000,1.0,0.578947,0.000000,0.000000,0.000000
13544,0.382353,0.177778,0.840426,0.000000,1.0,0.578947,0.066667,0.000000,0.000300
13545,0.352941,0.077778,0.851064,0.071429,1.0,0.578947,0.133333,0.000067,0.415467
...,...,...,...,...,...,...,...,...,...
16546,0.500000,0.411111,0.819149,0.214286,0.5,0.789474,0.200000,0.435333,0.795400
16547,0.529412,0.444444,0.744681,0.285714,1.0,0.789474,0.266667,0.795400,0.900500
16548,0.588235,0.566667,0.659574,0.571429,1.0,0.789474,0.333333,0.896833,0.924767
16549,0.647059,0.522222,0.574468,0.857143,0.5,0.789474,0.400000,0.924767,0.935967


In [15]:
X_test_scaled = x_scaler.transform(X_test)
print(X_test_scaled)
print(X_test_scaled.shape)

[[0.73529412 0.35555556 0.46808511 ... 0.53333333 0.95086667 0.95016667]
 [0.73529412 0.35555556 0.44680851 ... 0.6        0.95153333 0.95743333]
 [0.76470588 0.4        0.42553191 ... 0.66666667 0.95743333 0.8989    ]
 ...
 [0.47058824 0.         0.69148936 ... 0.26666667 0.58793333 0.767     ]
 [0.52941176 0.         0.57446809 ... 0.33333333 0.77796667 0.87186667]
 [0.64705882 0.         0.40425532 ... 0.46666667 0.8759     0.8551    ]]
(914, 9)


In [16]:
X_test_scaled_df = pd.DataFrame(X_test_scaled, index=X_test.index, columns=X_test.columns)
X_test_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
16551,0.735294,0.355556,0.468085,0.857143,0.25,0.736842,0.533333,0.950867,0.950167
16552,0.735294,0.355556,0.446809,0.642857,0.25,0.789474,0.600000,0.951533,0.957433
16553,0.764706,0.400000,0.425532,0.357143,0.00,0.789474,0.666667,0.957433,0.898900
16554,0.735294,0.488889,0.446809,0.214286,0.25,0.789474,0.733333,0.898900,0.878400
16555,0.705882,0.488889,0.500000,0.071429,1.00,0.789474,0.800000,0.877867,0.777867
...,...,...,...,...,...,...,...,...,...
18273,0.382353,0.000000,0.861702,0.071429,0.75,0.578947,0.133333,0.002233,0.243400
18274,0.411765,0.000000,0.819149,0.142857,0.75,0.631579,0.200000,0.245200,0.600467
18275,0.470588,0.000000,0.691489,0.285714,0.50,0.631579,0.266667,0.587933,0.767000
18276,0.529412,0.000000,0.574468,0.357143,0.50,0.578947,0.333333,0.777967,0.871867


In [17]:
x_scaller_all = MinMaxScaler().fit(X)
print(x_scaller_all)

MinMaxScaler()


In [18]:
X_scaled = x_scaller_all.transform(X)
print(X_scaled)
print(X_scaled.shape)

[[4.32432432e-01 7.77777778e-02 9.06250000e-01 ... 0.00000000e+00
  0.00000000e+00 0.00000000e+00]
 [4.32432432e-01 7.77777778e-02 9.37500000e-01 ... 6.66666667e-02
  0.00000000e+00 2.19498623e-04]
 [4.05405405e-01 5.55555556e-02 9.68750000e-01 ... 1.33333333e-01
  0.00000000e+00 1.86807341e-02]
 ...
 [4.32432432e-01 0.00000000e+00 6.97916667e-01 ... 2.66666667e-01
  5.87933333e-01 7.67000000e-01]
 [4.86486486e-01 0.00000000e+00 5.83333333e-01 ... 3.33333333e-01
  7.77966667e-01 8.71866667e-01]
 [5.94594595e-01 0.00000000e+00 4.16666667e-01 ... 4.66666667e-01
  8.75900000e-01 8.55100000e-01]]
(6090, 9)


In [19]:
X_scaled_df = pd.DataFrame(X_scaled, index=X.index, columns=X.columns)
X_scaled_df

,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Generacion_prev_hour,Generacion_prev_day
30,0.432432,0.077778,0.906250,0.000000,0.75,0.75,0.000000,0.000000,0.000000
31,0.432432,0.077778,0.937500,0.000000,0.50,0.80,0.066667,0.000000,0.000219
32,0.405405,0.055556,0.968750,0.000000,0.50,0.75,0.133333,0.000000,0.018681
33,0.432432,0.000000,0.927083,0.071429,0.25,0.80,0.200000,0.014627,0.257353
34,0.459459,0.000000,0.843750,0.142857,0.25,0.75,0.266667,0.196933,0.314437
...,...,...,...,...,...,...,...,...,...
18273,0.351351,0.000000,0.864583,0.071429,0.75,0.55,0.133333,0.002233,0.243400
18274,0.378378,0.000000,0.822917,0.142857,0.75,0.60,0.200000,0.245200,0.600467
18275,0.432432,0.000000,0.697917,0.285714,0.50,0.60,0.266667,0.587933,0.767000
18276,0.486486,0.000000,0.583333,0.357143,0.50,0.55,0.333333,0.777967,0.871867


In [20]:
y_scaler = MinMaxScaler().fit(y_train)
print(y_scaler)

MinMaxScaler()


In [21]:
y_train_scaled = y_scaler.transform(y_train)
print(y_train_scaled)
print(y_train_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.8077    ]
 [0.59663333]
 [0.11096667]]
(4263, 1)


In [22]:
y_train_scaled_df = pd.DataFrame(y_train_scaled, index=y_train.index, columns=y_train.columns)
y_train_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
13522,0.444233
13529,0.868733
13530,0.807700
13531,0.596633


In [23]:
y_val_scaled = y_scaler.transform(y_val)
print(y_val_scaled)
print(y_val_scaled.shape)

[[0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [6.66666667e-05]
 [1.89933333e-01]
 [6.10200000e-01]
 [7.91700000e-01]
 [8.05833333e-01]
 [7.12233333e-01]
 [6.85733333e-01]
 [6.81133333e-01]
 [7.68600000e-01]
 [9.80400000e-01]
 [7.70233333e-01]
 [4.56866667e-01]
 [1.07900000e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [3.56133333e-01]
 [4.50066667e-01]
 [6.51266667e-01]
 [9.75433333e-01]
 [9.28733333e-01]
 [9.71900000e-01]
 [9.31933333e-01]
 [5.67633333e-01]
 [6.32233333e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.00000000e-04]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.74833333e-01]
 [6.28000000e-01]
 [7.55433333e-01]
 [7.36500000e-01]
 [7.29200000e-01]
 [6.37000000e-01]
 [7.18033333e-01]
 [5.17733333e-01]
 [1.03866667e-01]
 [0.00000000e+00]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.66666667e-04]
 [0.00000000e+00]
 [6.66666667e-05]
 [5.93533333e-01]
 [6.18800000e-01]
 [8.85000000e-01]
 [8.953666

In [24]:
y_val_scaled_df = pd.DataFrame(y_val_scaled, index=y_val.index, columns=y_val.columns)
y_val_scaled_df

,Generación
13533,0.000000
13534,0.000000
13543,0.000000
13544,0.000067
13545,0.189933
...,...
16546,0.795400
16547,0.896833
16548,0.924767
16549,0.937500


In [25]:
y_test_scaled = y_scaler.transform(y_test)
print(y_test_scaled)
print(y_test_scaled.shape)

[[9.51533333e-01]
 [9.57433333e-01]
 [8.98900000e-01]
 [8.77866667e-01]
 [7.85400000e-01]
 [3.59033333e-01]
 [3.83666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.49000000e-02]
 [4.35333333e-01]
 [7.95400000e-01]
 [9.33400000e-01]
 [8.52400000e-01]
 [8.32100000e-01]
 [9.52200000e-01]
 [6.61300000e-01]
 [6.58233333e-01]
 [5.76366667e-01]
 [4.95800000e-01]
 [3.77400000e-01]
 [1.65566667e-01]
 [1.78666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [1.93000000e-02]
 [1.40800000e-01]
 [2.47800000e-01]
 [3.81700000e-01]
 [3.92633333e-01]
 [3.94533333e-01]
 [4.66566667e-01]
 [4.48600000e-01]
 [5.12533333e-01]
 [5.03500000e-01]
 [6.21466667e-01]
 [7.75366667e-01]
 [1.92633333e-01]
 [1.78666667e-02]
 [0.00000000e+00]
 [0.00000000e+00]
 [4.38333333e-02]
 [4.69166667e-01]
 [8.33500000e-01]
 [8.16000000e-01]
 [6.67466667e-01]
 [6.60733333e-01]
 [6.64166667e-01]
 [6.88000000e-01]
 [6.62833333e-01]
 [6.58766667e-01]
 [6.36333333e-01]
 [8.15333333e-01]
 [4.41033333e-01]
 [4.64000000e-02]
 [0.000000

In [26]:
y_test_scaled_df = pd.DataFrame(y_test_scaled, index=y_test.index, columns=y_test.columns)
y_test_scaled_df

,Generación
16551,0.951533
16552,0.957433
16553,0.898900
16554,0.877867
16555,0.785400
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


In [27]:
y_scaller_all = MinMaxScaler().fit(y)
print(y_scaller_all)

MinMaxScaler()


In [28]:
y_scaled = y_scaller_all.transform(y)
print(y_scaled)
print(y_scaled.shape)

[[0.        ]
 [0.        ]
 [0.01462717]
 ...
 [0.77796667]
 [0.87743333]
 [0.85326667]]
(6090, 1)


In [29]:
y_scaled_df = pd.DataFrame(y_scaled, index=y.index, columns=y.columns)
y_scaled_df

,Generación
30,0.000000
31,0.000000
32,0.014627
33,0.196933
34,0.167691
...,...
18273,0.245200
18274,0.587933
18275,0.777967
18276,0.877433


## Definición de modelos

### RandomForest

In [30]:
from lightgbm import LGBMRegressor
import optuna
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import cross_val_score
import seaborn as sns
from sklearn.metrics import mean_absolute_percentage_error as mean_absolute_percentage_error
from sklearn.metrics import mean_absolute_error as mean_absolute_error
from sklearn.metrics import mean_squared_error as mean_squared_error
from sklearn.metrics import r2_score as r2_score

In [31]:
# Inicializar listas para métricas
LightGBM_model = LGBMRegressor(num_leaves=500, subsample= 0.10698460631792395, colsample_bytree= 0.7272836809565294, min_data_in_leaf= 85)
LightGBM_model.fit(X_train_scaled_df, y_train_scaled_df)
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["LightGBM"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = LightGBM_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Warning] Found whitespace in feature_names, replace with underlines
[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.005456 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 756
[LightGBM] [Info] Number of data points in the train set: 4263, number of used features: 9
[LightGBM] [Info] Start training from score 0.312844
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further split

In [32]:
resultados

,LightGBM
16551,27911.718674
16552,28552.281566
16553,28173.508713
16554,25031.833853
16555,21975.28109
...,...
18273,5632.366084
18274,18901.861998
18275,24148.93331
18276,25572.670269


In [33]:
predicciones = y_test.copy()
predicciones

,Generación
16551,28546.0
16552,28723.0
16553,26967.0
16554,26336.0
16555,23562.0
...,...
18273,7356.0
18274,17638.0
18275,23339.0
18276,26323.0


In [34]:
predicciones["LightGBM"] = resultados["LightGBM"]
predicciones

,Generación,LightGBM
16551,28546.0,27911.718674
16552,28723.0,28552.281566
16553,26967.0,28173.508713
16554,26336.0,25031.833853
16555,23562.0,21975.28109
...,...,...
18273,7356.0,5632.366084
18274,17638.0,18901.861998
18275,23339.0,24148.93331
18276,26323.0,25572.670269


In [35]:
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")

MAE: 1420.5734
RMSE: 2441.7064
R²: 0.9516


## Random Forest

In [36]:
from sklearn.ensemble import RandomForestRegressor

In [37]:
#Modelo LightGBM
RF_model = RandomForestRegressor(
    criterion="squared_error",
    random_state=0,
    n_estimators=400,
    min_impurity_decrease=0,
    max_depth=None,
    bootstrap=True
)
RF_model.fit(X_train_scaled_df, y_train_scaled_df)
# Inicializar listas para métricas
resultados = pd.DataFrame(index = y_test_scaled_df.index, columns=["Random Forest"])
#Ciclo diario de predicción
for i in range(len(X_test)):
    inicio = i * 1
    fin = inicio + 1

    X_test_seg = X_test_scaled_df.iloc[inicio:fin, :]
    y_test_seg = y_test_scaled_df.iloc[inicio:fin]

    if len(X_test_seg) < 1:
        break

    y_pred = RF_model.predict(X_test_seg)
    y_pred = y_scaler.inverse_transform(y_pred.reshape(-1, 1))
    y_pred = np.clip(y_pred, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

    resultados.iloc[i, 0] = y_pred[0, 0]

In [38]:
predicciones["Random Forest"] = resultados["Random Forest"]
predicciones

,Generación,LightGBM,Random Forest
16551,28546.0,27911.718674,25071.187461
16552,28723.0,28552.281566,25834.987615
16553,26967.0,28173.508713,26222.885304
16554,26336.0,25031.833853,23317.3981
16555,23562.0,21975.28109,23437.167632
...,...,...,...
18273,7356.0,5632.366084,5438.219944
18274,17638.0,18901.861998,18263.627162
18275,23339.0,24148.93331,21918.767881
18276,26323.0,25572.670269,24646.911253


## Preparación redes neuronales

In [39]:
import numpy as np
import pandas as pd

def create_sliding_window_with_index(data_X, data_y, lookback):
    X, y, indices = [], [], []
    
    # Asegurar que `data_y` tiene los mismos índices que `data_X`
    data_y = data_y.reindex(data_X.index)

    max_index = len(data_X) - lookback

    for i in range(max_index):
        X.append(data_X.iloc[i:i + lookback].values)  # Ventana de entrada
        
        # Obtener el índice correcto en `data_y`
        y_index = data_X.index[i + lookback]

        # Extraer el valor correspondiente de `data_y`
        if y_index in data_y.index:
            y_value = data_y.loc[y_index]
            if isinstance(y_value, pd.Series):  # Si devuelve una serie, extraer el valor
                y_value = y_value.iloc[0]
        else:
            y_value = np.nan  # Si no está, asignamos NaN

        y.append(y_value)
        indices.append(y_index)  # 🔹 Guardamos el índice original de `data_y`

    # Convertimos `X` en un array y `y` en DataFrame conservando sus índices originales
    X_array = np.array(X)
    y_df = pd.DataFrame(y, index=indices, columns=['y'])  # 🔹 Conservamos los índices originales

    return X_array, y_df


In [40]:
lookback = 48  # Puedes ajustar a 24, 72, etc.

# Aplicar la ventana deslizante a cada conjunto
X_train_windowed, y_train_windowed = create_sliding_window_with_index(X_train_scaled_df, y_train_scaled_df, lookback)
X_val_windowed, y_val_windowed = create_sliding_window_with_index(X_val_scaled_df, y_val_scaled_df, lookback)
X_test_windowed, y_test_windowed = create_sliding_window_with_index(X_test_scaled_df, y_test_scaled_df, lookback)


In [41]:
print(f'X_train: {X_train_windowed.shape}, y_train: {y_train_windowed.shape}')
print(f'X_val: {X_val_windowed.shape}, y_val: {y_val_windowed.shape}')
print(f'X_test: {X_test_windowed.shape}, y_test: {y_test_windowed.shape}')

X_train: (4215, 48, 9), y_train: (4215, 1)
X_val: (865, 48, 9), y_val: (865, 1)
X_test: (866, 48, 9), y_test: (866, 1)


## CTNET

In [42]:
import tensorflow as tf
from tensorflow.keras import layers

In [43]:
def compile_and_fit(model, xtrain=X_train_windowed, ytrain=y_train_windowed, learning_rate=0.0001):
    model.compile(loss=[tf.keras.losses.MeanSquaredError()],
                  optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  metrics=[tf.keras.metrics.RootMeanSquaredError(), tf.keras.metrics.MeanAbsolutePercentageError(), tf.keras.metrics.MeanAbsoluteError()])
    
    history = model.fit(xtrain, ytrain, epochs=50,
                        batch_size=512, validation_split=0.2, verbose=1)
    return history

def Loss(train_loss, valid_loss):
    plt.plot(train_loss)
    plt.plot(valid_loss)
    plt.rcParams["figure.figsize"] = (15, 3)
    plt.title('Model Losses')
    plt.ylabel('Loss')
    plt.xlabel('Epoch')
    plt.legend(['Train Loss', 'Validation Loss'], loc='upper left')
    plt.savefig('out/loss_plot.png')
    plt.show()

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    x = layers.LayerNormalization()(inputs)
    x = layers.Conv1D(filters=ff_dim, kernel_size=1, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=128, kernel_size=2, activation="relu", padding="same")(x)
    x = layers.Conv1D(filters=inputs.shape[-1], kernel_size=1)(x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(norm_x, norm_x)
    res = x + inputs
    norm_x = layers.LayerNormalization()(res)
    return norm_x

def build_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = tf.keras.Input(shape=input_shape)
    x = inputs
    
    for _ in range(num_transformer_blocks):
        enc_out = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)
    
    x = layers.MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(enc_out, enc_out)
    res = x + enc_out
    x = layers.LayerNormalization(epsilon=1e-6)(res)
    x = layers.GlobalAveragePooling1D(data_format="channels_first")(x)
    x = layers.Dense(832, activation="relu")(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.Dropout(mlp_dropout)(x)

    outputs = layers.Dense(1)(x)
    
    return tf.keras.Model(inputs, outputs)

In [44]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=4, num_heads=3, ff_dim=32, num_transformer_blocks=3, mlp_units=[256], mlp_dropout=0.3, dropout=0.2)

In [45]:
history = compile_and_fit(CTNET)

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 127s 2s/step - loss: 0.2086 - mean_absolute_error: 0.3116 - mean_absolute_percentage_error: 918487.1250 - root_mean_squared_error: 0.4567 - val_loss: 0.1769 - val_mean_absolute_error: 0.2930 - val_mean_absolute_percentage_error: 4978389.5000 - val_root_mean_squared_error: 0.4206
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 16s 2s/step - loss: 0.2031 - mean_absolute_error: 0.3107 - mean_absolute_percentage_error: 5341677.0000 - root_mean_squared_error: 0.4506 - val_loss: 0.1664 - val_mean_absolute_error: 0.2897 - val_mean_absolute_percentage_error: 11264108.0000 - val_root_mean_squared_error: 0.4080
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 23s 2s/step - loss: 0.1897 - mean_absolute_error: 0.3041 - mean_absolute_percentage_error: 10926729.0000 - root_mean_squared_error: 0.4356 - val_loss: 0.1541 - val_mean_absolute_error: 0.2866 - val_mean_absolute_percentage_error: 19294226.0000 - val_root_mean_squared_error: 0.3926
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 29s 3s/step -

In [46]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

28/28 ━━━━━━━━━━━━━━━━━━━━ 16s 289ms/step


array([[ 6.08048558e-01],
       [ 6.17382824e-01],
       [ 6.25880361e-01],
       [ 5.97834527e-01],
       [ 5.84949851e-01],
       [ 4.80826020e-01],
       [ 1.82540044e-01],
       [ 3.76355015e-02],
       [-7.93174561e-03],
       [ 3.39207575e-02],
       [ 2.05353290e-01],
       [ 5.15641868e-01],
       [ 6.64154530e-01],
       [ 7.05802619e-01],
       [ 6.93364918e-01],
       [ 6.61562681e-01],
       [ 5.86813569e-01],
       [ 6.17465615e-01],
       [ 6.35779381e-01],
       [ 6.01564169e-01],
       [ 5.53250313e-01],
       [ 4.70212102e-01],
       [ 3.12266260e-01],
       [ 1.29592374e-01],
       [ 1.53262187e-02],
       [ 5.58611900e-02],
       [ 2.12805867e-01],
       [ 5.25018632e-01],
       [ 6.88778043e-01],
       [ 7.23066568e-01],
       [ 7.25709498e-01],
       [ 7.19139159e-01],
       [ 6.70661330e-01],
       [ 6.65848553e-01],
       [ 6.88022196e-01],
       [ 6.76537633e-01],
       [ 6.43085718e-01],
       [ 5.05868614e-01],
       [ 3.0

In [47]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [48]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [49]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
16551,28546.0,27911.718674,25071.187461,NaN
16552,28723.0,28552.281566,25834.987615,NaN
16553,26967.0,28173.508713,26222.885304,NaN
16554,26336.0,25031.833853,23317.3981,NaN
16555,23562.0,21975.28109,23437.167632,NaN
...,...,...,...,...
18273,7356.0,5632.366084,5438.219944,6833.204102
18274,17638.0,18901.861998,18263.627162,14250.267578
18275,23339.0,24148.93331,21918.767881,17568.277344
18276,26323.0,25572.670269,24646.911253,20057.130859


In [50]:
predicciones["CTNET"] = predicciones["CTNET"].fillna(0)

In [51]:
# import optuna
# import tensorflow as tf
# from tensorflow.keras import layers
# from sklearn.model_selection import train_test_split

# # Definir la función objetivo para Optuna
# def objective(trial):
#     # Sugerir valores para los hiperparámetros
#     head_size = trial.suggest_int("head_size", 8, 64, step=8)
#     num_heads = trial.suggest_int("num_heads", 2, 8, step=2)
#     ff_dim = trial.suggest_int("ff_dim", 32, 256, step=32)
#     num_transformer_blocks = trial.suggest_int("num_transformer_blocks", 1, 4)
#     mlp_units = trial.suggest_categorical("mlp_units", [[128, 64], [256, 128, 64], [512, 256, 128]])
#     dropout = trial.suggest_float("dropout", 0.1, 0.5, step=0.1)
#     mlp_dropout = trial.suggest_float("mlp_dropout", 0.1, 0.5, step=0.1)
#     learning_rate = trial.suggest_loguniform("learning_rate", 1e-5, 1e-2)

#     # Construcción del modelo con los hiperparámetros sugeridos
#     model = build_model(
#         input_shape=X_train_windowed.shape[1:],
#         head_size=head_size,
#         num_heads=num_heads,
#         ff_dim=ff_dim,
#         num_transformer_blocks=num_transformer_blocks,
#         mlp_units=mlp_units,
#         dropout=dropout,
#         mlp_dropout=mlp_dropout
#     )

#     # Compilar el modelo con los hiperparámetros sugeridos
#     model.compile(
#         loss=tf.keras.losses.MeanSquaredError(),
#         optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
#         metrics=[tf.keras.metrics.RootMeanSquaredError()]
#     )

#     # Entrenamiento con un número reducido de épocas para acelerar la búsqueda
#     history = model.fit(
#         X_train_windowed, y_train_windowed,
#         validation_split=0.2,
#         epochs=50,  # Reducimos las épocas para acelerar la búsqueda
#         batch_size=512,
#         verbose=0
#     )

#     # Obtener la métrica de validación (RMSE) y minimizarla
#     val_rmse = min(history.history["val_root_mean_squared_error"])
    
#     return val_rmse  # Queremos minimizar el RMSE

# # Ejecutar la optimización de hiperparámetros
# study = optuna.create_study(direction="minimize")
# study.optimize(objective, n_trials=20, timeout=3600)  # 20 iteraciones, máximo 1 hora

# # Mostrar los mejores hiperparámetros encontrados
# best_params = study.best_params
# print(f"Mejores hiperparámetros: {best_params}")


In [52]:
CTNET = build_model((X_train_windowed.shape[1], X_train_windowed.shape[2]), head_size=16, num_heads=8, ff_dim=256, num_transformer_blocks=1, mlp_units=[128,64], mlp_dropout=0.2, dropout=0.2)

In [53]:
history = compile_and_fit(CTNET, learning_rate = 0.0010762230908145116)

Epoch 1/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 183s 8s/step - loss: 0.1925 - mean_absolute_error: 0.3077 - mean_absolute_percentage_error: 10932750.0000 - root_mean_squared_error: 0.4385 - val_loss: 0.0979 - val_mean_absolute_error: 0.2837 - val_mean_absolute_percentage_error: 90615656.0000 - val_root_mean_squared_error: 0.3129
Epoch 2/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 71s 5s/step - loss: 0.1229 - mean_absolute_error: 0.3154 - mean_absolute_percentage_error: 100067488.0000 - root_mean_squared_error: 0.3505 - val_loss: 0.0976 - val_mean_absolute_error: 0.2863 - val_mean_absolute_percentage_error: 99219888.0000 - val_root_mean_squared_error: 0.3123
Epoch 3/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 50s 6s/step - loss: 0.1179 - mean_absolute_error: 0.3004 - mean_absolute_percentage_error: 76175072.0000 - root_mean_squared_error: 0.3433 - val_loss: 0.0991 - val_mean_absolute_error: 0.2821 - val_mean_absolute_percentage_error: 83843536.0000 - val_root_mean_squared_error: 0.3148
Epoch 4/50
7/7 ━━━━━━━━━━━━━━━━━━━━ 88s 6s/s

In [54]:
CTNET_predictions = CTNET.predict(X_test_windowed)
CTNET_predictions

28/28 ━━━━━━━━━━━━━━━━━━━━ 30s 432ms/step


array([[ 7.46214509e-01],
       [ 6.49773002e-01],
       [ 4.69265759e-01],
       [ 7.98773408e-01],
       [ 8.21251273e-01],
       [ 8.07953596e-01],
       [ 7.32283741e-02],
       [ 4.81670350e-02],
       [ 1.80917792e-02],
       [ 6.26400113e-02],
       [ 2.09366933e-01],
       [ 3.85104358e-01],
       [ 5.64345598e-01],
       [ 5.70777774e-01],
       [ 6.76092386e-01],
       [ 6.74589396e-01],
       [ 3.67603481e-01],
       [ 2.92470157e-01],
       [ 6.13041103e-01],
       [ 7.05772758e-01],
       [ 5.85287094e-01],
       [ 3.36795270e-01],
       [ 5.33193164e-02],
       [ 9.29204747e-03],
       [ 3.28700095e-02],
       [ 4.66286056e-02],
       [ 2.58020163e-01],
       [ 5.13337314e-01],
       [ 6.00018740e-01],
       [ 4.49948549e-01],
       [ 5.79623103e-01],
       [ 8.78267169e-01],
       [ 7.71459162e-01],
       [ 7.01574564e-01],
       [ 8.22427392e-01],
       [ 8.15851927e-01],
       [ 6.59103453e-01],
       [ 2.71429002e-01],
       [ 4.0

In [55]:
CTNET_predictions = y_scaler.inverse_transform(CTNET_predictions.reshape(-1, 1))
CTNET_predictions = np.clip(CTNET_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [56]:
resultados = pd.DataFrame(CTNET_predictions, index = y_test_windowed.index, columns=["CTNET"])

In [57]:
predicciones["CTNET"] = resultados["CTNET"]
predicciones

,Generación,LightGBM,Random Forest,CTNET
16551,28546.0,27911.718674,25071.187461,NaN
16552,28723.0,28552.281566,25834.987615,NaN
16553,26967.0,28173.508713,26222.885304,NaN
16554,26336.0,25031.833853,23317.3981,NaN
16555,23562.0,21975.28109,23437.167632,NaN
...,...,...,...,...
18273,7356.0,5632.366084,5438.219944,3277.965576
18274,17638.0,18901.861998,18263.627162,30000.000000
18275,23339.0,24148.93331,21918.767881,17159.171875
18276,26323.0,25572.670269,24646.911253,12295.125000


## Forecasting

In [58]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import *
from tensorflow.keras.callbacks import ModelCheckpoint
from tensorflow.keras.losses import MeanSquaredError
from tensorflow.keras.metrics import RootMeanSquaredError
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.losses import Huber
from tensorflow.keras.callbacks import EarlyStopping

In [59]:
Forecast_model = Sequential()
Forecast_model.add(InputLayer((X_train_windowed.shape[1], X_train_windowed.shape[2])))

#CNN
Forecast_model.add(Conv1D(filters=64, kernel_size=2, padding='same', activation='relu'))
Forecast_model.add(BatchNormalization())  # 🔹 Nueva Normalización aquí
Forecast_model.add(MaxPooling1D(pool_size=2))

#model_Soleado.add(Flatten())
#BiLSTM
Forecast_model.add(Bidirectional(LSTM(128, return_sequences=True)))
Forecast_model.add(Bidirectional(LSTM(64, return_sequences=True)))
Forecast_model.add(Dropout(0.2))  # 🔹 Mayor regularización en BiLSTM
Forecast_model.add(Bidirectional(LSTM(32, return_sequences=False)))

#Normalización y Dropout
Forecast_model.add(BatchNormalization())
Forecast_model.add(Dropout(0.3))

# Capas Densas
Forecast_model.add(Dense(16, activation='relu'))
Forecast_model.add(Dense(1, 'relu'))

Forecast_model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_12 (Conv1D)              │ (None, 48, 64)         │         1,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 64)         │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 24, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional (Bidirectional)   │ (None, 24, 256)        │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 24, 128)        │       164,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_8 (Dropout)             │ (None, 24, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 64)             │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_8 (Dense)                 │ (None, 16)             │         1,040 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 405,985 (1.55 MB)

 Trainable params: 405,729 (1.55 MB)

 Non-trainable params: 256 (1.00 KB)

In [60]:
cp = ModelCheckpoint('Forcasting_model.keras', save_best_only=True)
Forecast_model.compile(optimizer=Adam(learning_rate=0.0001), loss=Huber(delta=1000), metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [61]:
history = Forecast_model.fit(X_train_windowed, y_train_windowed, validation_data=(X_val_windowed, y_val_windowed), epochs=100, batch_size=8, callbacks=[cp, early_stop])

Epoch 1/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 613s 593ms/step - loss: 0.1061 - mae: 0.3167 - val_loss: 0.1543 - val_mae: 0.4211
Epoch 2/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 244s 451ms/step - loss: 0.1048 - mae: 0.3155 - val_loss: 0.1504 - val_mae: 0.4170
Epoch 3/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 279s 472ms/step - loss: 0.0996 - mae: 0.3041 - val_loss: 0.1385 - val_mae: 0.3922
Epoch 4/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 268s 470ms/step - loss: 0.1008 - mae: 0.3074 - val_loss: 0.1395 - val_mae: 0.3975
Epoch 5/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 273s 479ms/step - loss: 0.0891 - mae: 0.2810 - val_loss: 0.0760 - val_mae: 0.2842
Epoch 6/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 226s 397ms/step - loss: 0.0863 - mae: 0.2806 - val_loss: 0.0898 - val_mae: 0.3007
Epoch 7/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 256s 379ms/step - loss: 0.0726 - mae: 0.2525 - val_loss: 0.0411 - val_mae: 0.1964
Epoch 8/100
527/527 ━━━━━━━━━━━━━━━━━━━━ 202s 372ms/step - loss: 0.0618 - mae: 0.2361 - val_loss: 0.0429 - val_mae: 0.2116
Epoch 9/100
527/

In [62]:
Forecast_predictions = Forecast_model.predict(X_test_windowed)
Forecast_predictions

28/28 ━━━━━━━━━━━━━━━━━━━━ 38s 824ms/step


array([[7.1311688e-01],
       [7.0359111e-01],
       [6.9095010e-01],
       [6.3197130e-01],
       [5.9145850e-01],
       [4.6305358e-01],
       [0.0000000e+00],
       [0.0000000e+00],
       [0.0000000e+00],
       [0.0000000e+00],
       [1.9201040e-01],
       [6.3998175e-01],
       [6.6377026e-01],
       [6.6325420e-01],
       [6.5615195e-01],
       [6.6877878e-01],
       [6.5890861e-01],
       [6.8396306e-01],
       [6.2228876e-01],
       [5.5953974e-01],
       [4.9951997e-01],
       [5.4587513e-01],
       [2.1028824e-02],
       [0.0000000e+00],
       [0.0000000e+00],
       [0.0000000e+00],
       [2.1160601e-01],
       [6.8799460e-01],
       [6.9395137e-01],
       [7.0801079e-01],
       [6.8318254e-01],
       [6.8450898e-01],
       [7.3837411e-01],
       [7.6834816e-01],
       [7.7653617e-01],
       [7.3331827e-01],
       [6.3479710e-01],
       [4.2257011e-01],
       [0.0000000e+00],
       [0.0000000e+00],
       [0.0000000e+00],
       [0.000000

In [63]:
Forecast_predictions = y_scaler.inverse_transform(Forecast_predictions.reshape(-1, 1))
Forecast_predictions = np.clip(Forecast_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [64]:
Forecast_resultados = pd.DataFrame(Forecast_predictions, index = y_test_windowed.index, columns=["Forecast"])

In [65]:
predicciones["Forecast"] = Forecast_resultados["Forecast"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast
16551,28546.0,27911.718674,25071.187461,NaN,NaN
16552,28723.0,28552.281566,25834.987615,NaN,NaN
16553,26967.0,28173.508713,26222.885304,NaN,NaN
16554,26336.0,25031.833853,23317.3981,NaN,NaN
16555,23562.0,21975.28109,23437.167632,NaN,NaN
...,...,...,...,...,...
18273,7356.0,5632.366084,5438.219944,3277.965576,0.000000
18274,17638.0,18901.861998,18263.627162,30000.000000,18738.458984
18275,23339.0,24148.93331,21918.767881,17159.171875,19559.654297
18276,26323.0,25572.670269,24646.911253,12295.125000,18438.984375


## Métricas

In [66]:
predicciones.loc[~predicciones['CTNET'].isna(),'Generación']

16623    20640.0
16624    19885.0
16625    19763.0
16626    19090.0
16627    24460.0
          ...   
18273     7356.0
18274    17638.0
18275    23339.0
18276    26323.0
18278    25598.0
Name: Generación, Length: 866, dtype: float64

## Photovoltaic

In [67]:
from tensorflow.keras.models import Model
inputs = Input(shape=(X_train_windowed.shape[1], X_train_windowed.shape[2]))

# Primera capa CNN
x = Conv1D(filters=64, kernel_size=4, padding='same', activation='relu')(inputs)
x = MaxPooling1D(pool_size=2)(x)

# Segunda capa CNN
x = Conv1D(filters=128, kernel_size=4, padding='same', activation='relu')(x)
x = MaxPooling1D(pool_size=2)(x)

# Capa BiGRU
x = Bidirectional(GRU(64, return_sequences=True))(x)

# Atención: se define de forma explícita
attention = MultiHeadAttention(num_heads=4, key_dim=128)(x, x)

# Aplanar y agregar Dropout
x = Flatten()(attention)
x = Dropout(0.4)(x)
initializer = tf.keras.initializers.HeNormal()
x = Dense(64, activation="relu", kernel_regularizer=l2(0.01))(x)
x = Dense(32, activation="relu")(x)  # Otra capa intermedia

# Capa de salida
outputs = Dense(1, activation="linear")(x)

# Definir el modelo
Photo_model = Model(inputs=inputs, outputs=outputs)

# Resumen del modelo
Photo_model.summary()

Model: "functional_13"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 48, 9)     │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_13 (Conv1D)  │ (None, 48, 64)    │      2,368 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 24, 64)    │          0 │ conv1d_13[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_14 (Conv1D)  │ (None, 24, 128)   │     32,896 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_2     │ (None, 12, 128)   │          0 │ conv1d_14[0][0]   │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ bidirectional_3     │ (None, 12, 128)   │     74,496 │ max_pooling1d_2[… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 12, 128)   │    263,808 │ bidirectional_3[… │
│ (MultiHeadAttentio… │                   │            │ bidirectional_3[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 1536)      │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 1536)      │          0 │ flatten[0][0]     │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 64)        │     98,368 │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 32)        │      2,080 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 1)         │         33 │ dense_11[0][0]    │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 474,049 (1.81 MB)

 Trainable params: 474,049 (1.81 MB)

 Non-trainable params: 0 (0.00 B)

In [68]:
cp2 = ModelCheckpoint('Photovoltaic_model.keras', save_best_only=True)
Photo_model.compile(optimizer=Adam(learning_rate=0.0001), loss="mean_squared_error", metrics=['mae'])
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

In [69]:
history = Photo_model.fit(
    X_train_windowed, y_train_windowed,
    validation_data=(X_val_windowed, y_val_windowed),
    epochs=50,
    batch_size=16,
    callbacks=[cp, early_stop]
)

Epoch 1/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 217s 189ms/step - loss: 1.0846 - mae: 0.2955 - val_loss: 0.5440 - val_mae: 0.3208
Epoch 2/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 70s 132ms/step - loss: 0.4198 - mae: 0.2647 - val_loss: 0.2251 - val_mae: 0.2117
Epoch 3/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 33s 112ms/step - loss: 0.1680 - mae: 0.1744 - val_loss: 0.1180 - val_mae: 0.1853
Epoch 4/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 47s 130ms/step - loss: 0.0950 - mae: 0.1632 - val_loss: 0.0823 - val_mae: 0.1726
Epoch 5/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 41s 139ms/step - loss: 0.0654 - mae: 0.1492 - val_loss: 0.0822 - val_mae: 0.1805
Epoch 6/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 51s 167ms/step - loss: 0.0541 - mae: 0.1420 - val_loss: 0.0643 - val_mae: 0.1610
Epoch 7/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 61s 82ms/step - loss: 0.0467 - mae: 0.1336 - val_loss: 0.0582 - val_mae: 0.1563
Epoch 8/50
264/264 ━━━━━━━━━━━━━━━━━━━━ 49s 110ms/step - loss: 0.0446 - mae: 0.1323 - val_loss: 0.0537 - val_mae: 0.1482
Epoch 9/50
264/264 ━━━━━━━━━━━━━

In [70]:
Photo_predictions = Photo_model.predict(X_test_windowed)
Photo_predictions

28/28 ━━━━━━━━━━━━━━━━━━━━ 30s 519ms/step


array([[0.8722664 ],
       [0.8610749 ],
       [0.8691761 ],
       [0.821877  ],
       [0.71374404],
       [0.47738907],
       [0.05807815],
       [0.0173207 ],
       [0.01323837],
       [0.02024901],
       [0.20001665],
       [0.7839481 ],
       [0.9083557 ],
       [0.83799547],
       [0.7986792 ],
       [0.86305135],
       [0.9068074 ],
       [0.98112106],
       [0.8415758 ],
       [0.8062389 ],
       [0.7197869 ],
       [0.67489254],
       [0.11282066],
       [0.01392541],
       [0.01104676],
       [0.01503696],
       [0.20246555],
       [0.8470067 ],
       [0.9547308 ],
       [0.91959274],
       [0.85805154],
       [0.87668455],
       [0.8636388 ],
       [0.8943397 ],
       [0.83794045],
       [0.7222346 ],
       [0.59073794],
       [0.34947363],
       [0.04083082],
       [0.01950001],
       [0.01458783],
       [0.02578225],
       [0.3405595 ],
       [0.84788346],
       [0.9119539 ],
       [0.92665935],
       [0.9452731 ],
       [0.914

In [71]:
Photo_predictions = y_scaler.inverse_transform(Photo_predictions.reshape(-1, 1))
Photo_predictions = np.clip(Photo_predictions, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000

In [72]:
Photo_resultados = pd.DataFrame(Photo_predictions, index = y_test_windowed.index, columns=["Photo"])

In [73]:
predicciones["Photo"] = Photo_resultados["Photo"]
predicciones

,Generación,LightGBM,Random Forest,CTNET,Forecast,Photo
16551,28546.0,27911.718674,25071.187461,NaN,NaN,NaN
16552,28723.0,28552.281566,25834.987615,NaN,NaN,NaN
16553,26967.0,28173.508713,26222.885304,NaN,NaN,NaN
16554,26336.0,25031.833853,23317.3981,NaN,NaN,NaN
16555,23562.0,21975.28109,23437.167632,NaN,NaN,NaN
...,...,...,...,...,...,...
18273,7356.0,5632.366084,5438.219944,3277.965576,0.000000,2902.767578
18274,17638.0,18901.861998,18263.627162,30000.000000,18738.458984,17411.048828
18275,23339.0,24148.93331,21918.767881,17159.171875,19559.654297,15489.687500
18276,26323.0,25572.670269,24646.911253,12295.125000,18438.984375,15787.542969


In [74]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['LightGBM'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['LightGBM']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones['Generación'], predicciones['Random Forest'])):.4f}")
print(f"R²: {r2_score(predicciones['Generación'], predicciones['Random Forest']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['CTNET'].isna(),'Generación'], predicciones.loc[~predicciones['CTNET'].isna(),'CTNET']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Forecast'].isna(),'Generación'], predicciones.loc[~predicciones['Forecast'].isna(),'Forecast']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo'])):.4f}")
print(f"R²: {r2_score(predicciones.loc[~predicciones['Photo'].isna(),'Generación'], predicciones.loc[~predicciones['Photo'].isna(),'Photo']):.4f}")

LightGBM
MAE: 1420.5734
RMSE: 2441.7064
R²: 0.9516
Random Forest
MAE: 1458.7094
RMSE: 2539.9484
R²: 0.9476
CTNET
MAE: 4970.8306
RMSE: 7254.2356
R²: 0.5778
Forecast
MAE: 3506.8802
RMSE: 5228.1843
R²: 0.7807
Photovoltaic
MAE: 3283.4414
RMSE: 4945.0750
R²: 0.8038


In [75]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones.loc[:, "LightGBM":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,Cluster GMM,Generacion_prev_hour,Generacion_prev_day,LightGBM,Random Forest,CTNET,Forecast,Photo
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,Noche,0.0,0.0,NaN,NaN,NaN,NaN,NaN


## X_train para hacer análisis de sobreajuste

In [76]:
predicciones_train = y_train.copy()

In [77]:
LightGBM_predictions_train = LightGBM_model.predict(X_train_scaled_df)
LightGBM_predictions_train = y_scaler.inverse_transform(LightGBM_predictions_train.reshape(-1, 1))
LightGBM_predictions_train = np.clip(LightGBM_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
LightGBM_resultados = pd.DataFrame(LightGBM_predictions_train, index = y_train_scaled_df.index, columns=["LightGBM_train"])
predicciones_train["LightGBM_train"] = LightGBM_resultados["LightGBM_train"]

[LightGBM] [Warning] min_data_in_leaf is set=85, min_child_samples=20 will be ignored. Current value: min_data_in_leaf=85


In [78]:
RandomForest_predictions_train = RF_model.predict(X_train_scaled_df)
RandomForest_predictions_train = y_scaler.inverse_transform(RandomForest_predictions_train.reshape(-1, 1))
RandomForest_predictions_train = np.clip(RandomForest_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
RandomForest_resultados = pd.DataFrame(RandomForest_predictions_train, index = y_train_scaled_df.index, columns=["RandomForest_train"])
predicciones_train["RandomForest_train"] = RandomForest_resultados["RandomForest_train"]

In [79]:
CTNET_predictions_train = CTNET.predict(X_train_windowed)
CTNET_predictions_train = y_scaler.inverse_transform(CTNET_predictions_train.reshape(-1, 1))
CTNET_predictions_train = np.clip(CTNET_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
CTNET_resultados = pd.DataFrame(CTNET_predictions_train, index = y_train_windowed.index, columns=["CTNET_train"])
predicciones_train["CTNET_train"] = CTNET_resultados["CTNET_train"]

132/132 ━━━━━━━━━━━━━━━━━━━━ 15s 78ms/step


In [80]:
Forecast_predictions_train = Forecast_model.predict(X_train_windowed)
Forecast_predictions_train = y_scaler.inverse_transform(Forecast_predictions_train.reshape(-1, 1))
Forecast_predictions_train = np.clip(Forecast_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Forecast_resultados = pd.DataFrame(Forecast_predictions_train, index = y_train_windowed.index, columns=["Forecast_train"])
predicciones_train["Forecast_train"] = Forecast_resultados["Forecast_train"]

132/132 ━━━━━━━━━━━━━━━━━━━━ 13s 83ms/step


In [81]:
Photo_predictions_train = Photo_model.predict(X_train_windowed)
Photo_predictions_train = y_scaler.inverse_transform(Photo_predictions_train.reshape(-1, 1))
Photo_predictions_train = np.clip(Photo_predictions_train, 0, 30000)  # 🔹 Limitamos entre 0 y 30,000
Photo_resultados = pd.DataFrame(Photo_predictions_train, index = y_train_windowed.index, columns=["Photo_train"])
predicciones_train["Photo_train"] = Photo_resultados["Photo_train"]

132/132 ━━━━━━━━━━━━━━━━━━━━ 5s 34ms/step


In [82]:
predicciones_train

,Generación,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
30,0.000000,23.532782,0.000000,NaN,NaN,NaN
31,0.000000,40.239559,30.457186,NaN,NaN,NaN
32,438.814997,435.553357,607.033307,NaN,NaN,NaN
33,5908.000884,9489.346757,6944.263279,NaN,NaN,NaN
34,5030.740421,8385.395332,7066.862453,NaN,NaN,NaN
...,...,...,...,...,...,...
13522,13327.000000,18687.064058,16171.792612,22546.242188,17675.566406,18765.146484
13529,26062.000000,25593.269947,25721.809674,17246.470703,16239.867188,16280.187500
13530,24231.000000,23192.748917,23637.722526,26464.271484,20261.277344,19342.685547
13531,17899.000000,16396.132314,16887.833162,24251.800781,15183.484375,15574.723633


In [83]:
print("LightGBM")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['LightGBM_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['LightGBM_train']):.4f}")
print("Random Forest")
print(f"MAE: {mean_absolute_error(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train['Generación'], predicciones_train['RandomForest_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train['Generación'], predicciones_train['RandomForest_train']):.4f}")
print("CTNET")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['CTNET_train'].isna(),'CTNET_train']):.4f}")
print("Forecast")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Forecast_train'].isna(),'Forecast_train']):.4f}")
print("Photovoltaic")
print(f"MAE: {mean_absolute_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train'])):.4f}")
print(f"R²: {r2_score(predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Generación'], predicciones_train.loc[~predicciones_train['Photo_train'].isna(),'Photo_train']):.4f}")

LightGBM
MAE: 1185.4238
RMSE: 2135.2772
R²: 0.9545
Random Forest
MAE: 503.4450
RMSE: 968.4935
R²: 0.9906
CTNET
MAE: 2137.7970
RMSE: 3847.7656
R²: 0.8519
Forecast
MAE: 3588.5238
RMSE: 5638.7994
R²: 0.6819
Photovoltaic
MAE: 3377.7980
RMSE: 5165.0556
R²: 0.7331


In [84]:
# Seleccionar las columnas desde "LightGBM" en adelante
columnas_nuevas = predicciones_train.loc[:, "LightGBM_train":]

# Unir con `datos` usando el índice, manteniendo todo en `datos`
datos = datos.merge(columnas_nuevas, left_index=True, right_index=True, how='left')

# Ver resultado
datos.head()


,Fecha,Generación,Temperatura,Probabilidad Lluvia,Humedad Relativa,Índice UV,Condición Cielo,DPT,Hora,Cluster KMeans,...,LightGBM,Random Forest,CTNET,Forecast,Photo,LightGBM_train,RandomForest_train,CTNET_train,Forecast_train,Photo_train
24,2022-09-02 00:00:00,0.0,19,6,76,0,4,15,0,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
25,2022-09-02 01:00:00,0.0,18,7,81,0,4,15,1,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
26,2022-09-02 02:00:00,0.0,18,7,84,0,4,15,2,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
27,2022-09-02 03:00:00,0.0,18,7,86,0,4,15,3,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
28,2022-09-02 04:00:00,0.0,17,7,86,0,4,15,4,Noche,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [85]:
datos.to_excel("04.6_Predicciones_Conjunto_lluvioso GMM.xlsx", index=True)

## Guardamos los modelos

In [86]:
import joblib

# Guardar modelo LightGBM
joblib.dump(LightGBM_model, "4_6_LightGBM_model.pkl")

# Guardar modelo Random Forest
joblib.dump(RF_model, "4_6_RandomForest_model.pkl")


['4_6_RandomForest_model.pkl']

In [87]:
CTNET.save("4_6_CTNET_model.keras")
Forecast_model.save("4_6_Forecast_model.keras")
Photo_model.save("4_6_Photo_model.keras")